# 11. Container With Most Water
**Difficulty:** 🟡 Medium · **Topic:** Array · **LeetCode:** https://leetcode.com/problems/container-with-most-water/

## 💡 Concepts

**Core concept(s):** **Two pointers** converging from both ends, driven by a greedy observation.

**Why it applies here:** Water held between lines `i` and `j` is `min(height[i], height[j]) * (j - i)` — limited by the **shorter** wall. Starting at the widest gap (both ends), the only way to possibly gain area as width shrinks is to raise the limiting (shorter) wall. So we always move the shorter pointer inward; moving the taller one can never help.

**Key intuition / mental model:** Widest first. Area is capped by the shorter side, so discard the shorter side and hope for a taller one — each step provably can't skip a better answer.

---

### 📚 What is the Two-Pointer Technique?
Two indices sweeping toward each other in a **single O(n) pass**, using **O(1)** space. A rule (here: "move the shorter wall") decides which pointer advances, so we never need to test all pairs.

### 📚 Why is moving the shorter wall safe (greedy proof sketch)?
The container is bounded by the shorter wall. Keeping the shorter wall and shrinking width can only *decrease* area, so no better container uses that shorter wall at this width — it's safe to discard it.

## 📝 Problem

Given `height[]`, pick two lines that with the x-axis form a container holding the most water. Return that maximum area.

**Example**
```
Input:  height = [1, 8, 6, 2, 5, 4, 8, 3, 7]
Output: 49       # between indices 1 and 8: min(8,7) * (8-1) = 49
```
**Constraints:** `2 <= len(height) <= 10^5`.

> Two meaningfully distinct approaches: O(n²) brute force and the O(n) two-pointer optimal.

### Approach 1 — Brute Force (worst)

**Idea:** Try every pair of lines, compute area, keep the max.

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def max_area_brute(height: List[int]) -> int:
    best = 0
    n = len(height)
    for i in range(n):                     # try every pair of walls (i, j)
        for j in range(i + 1, n):
            area = min(height[i], height[j]) * (j - i)  # water = shorter wall x width
            best = max(best, area)
    return best

### Approach 2 — Two Pointers (optimal)

**Idea:** Start at both ends. Compute area, then move the pointer at the **shorter** wall inward (the only move that can increase area).

**Time complexity:** `O(n)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def max_area_optimal(height: List[int]) -> int:
    lo, hi = 0, len(height) - 1            # start with the two widest walls
    best = 0
    while lo < hi:
        area = min(height[lo], height[hi]) * (hi - lo)  # water is capped by the shorter wall
        best = max(best, area)
        if height[lo] < height[hi]:        # move the SHORTER wall inward...
            lo += 1                        # ...it's the only move that can improve the area
        else:
            hi -= 1
    return best

In [ ]:
# Correctness check
tests = [
    ([1, 8, 6, 2, 5, 4, 8, 3, 7], 49),
    ([1, 1], 1),
    ([4, 3, 2, 1, 4], 16),
    ([1, 2, 1], 2),
]
for height, expected in tests:
    b, o = max_area_brute(height), max_area_optimal(height)
    print(f"{height} -> brute={b}, optimal={o} | expected={expected}")
    assert b == o == expected, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit) so the measurement reflects the true bound. Sub-millisecond rows are noisy — look at the trend, not one number.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    height = list(range(1, n + 1))         # brute always scans all pairs
    return (height,)

solutions = {
    "brute   O(n^2)": max_area_brute,
    "optimal O(n)  ": max_area_optimal,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Converging two pointers with a greedy move:** When a value is bounded by the *worse* of two ends, discard that end — turns O(n²) pair search into O(n).
- **Start from the extreme:** Beginning at the widest/largest configuration often makes the greedy step obvious.
- **Signal to reach for it:** "max area/…​ between two indices", "pair optimizing a min/max × distance", sorted-or-symmetric pair problems.
- **Related problems:** Trapping Rain Water, 3Sum, Two Sum II, Valid Palindrome.
- **Common pitfalls:** (1) moving the taller wall (can't improve); (2) moving both pointers on a tie unnecessarily (moving either is fine); (3) using width `hi-lo+1` instead of `hi-lo`.